# 💻 Notebook do Aluno — Aula 04: Context Engineering De prompt para contexto

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 04/14 — Módulo 1: LangChain Foundations · 🏁 Entrega CKP01**  
**⏱️ 1h40min**  
**🧠 Context rot · XML tagging · Meta prompting**  
**🏁 CKP01 entrega**  

---

## 🎯 Objetivo da aula

Entender que o contexto é um recurso finito e caro — e aprender a preenchê-lo intencionalmente: certas informações no lugar certo, na hora certa, com o mínimo de tokens necessário. Aplicar isso ao CKP01.

---

## Como usar este notebook

- Rode as células **na ordem**, de cima para baixo (`Shift+Enter`).
- Complete apenas as partes marcadas com `___` e `👉 LACUNA`.
- Não apague o código já pronto — ele é o andaime do lab.
- Salve sua cópia: **Arquivo > Salvar uma cópia no Drive**.

---

## 🧩 Andaime da aula — complete as lacunas

Complete as lacunas marcadas com `___`.

In [ ]:
!pip install langchain langchain-ollama tiktoken -q

import tiktoken
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from google.colab import userdata
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
llm = ChatOllama(model="gpt-oss:120b")

def contar_tokens(texto: str, modelo: str = "gpt-4") -> int:
    return len(tiktoken.encoding_for_model(modelo).encode(texto))

# 👉 LACUNA 1: cole aqui o system prompt original do grupo (Aulas 01–03)
SYSTEM_ORIGINAL = ___

# 👉 LACUNA 2: reescreva com XML tagging — persona, restricoes, formato
SYSTEM_OTIMIZADO = ___  # use <persona>, <restricoes>, <formato>

# Medição automática
tok_antes  = contar_tokens(SYSTEM_ORIGINAL)
tok_depois = contar_tokens(SYSTEM_OTIMIZADO)
print(f"Antes:  {tok_antes} tokens")
print(f"Depois: {tok_depois} tokens")
print(f"Redução: {(1-tok_depois/tok_antes)*100:.1f}%")

# 👉 LACUNA 3: crie 2 chains (antes e depois) e teste com a mesma pergunta
chain_antes  = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_ORIGINAL),
    ("human",  ___),
]) | llm | StrOutputParser()

chain_depois = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_OTIMIZADO),
    ("human",  ___),
]) | llm | StrOutputParser()

# 👉 LACUNA 4: invoke com a mesma pergunta de teste para comparar saídas
pergunta_teste = ___
print("=== ANTES ===\n", chain_antes.invoke({___: pergunta_teste}))
print("=== DEPOIS ===\n", chain_depois.invoke({___: pergunta_teste}))

---

## ✍️ Suas anotações

Registre aqui as observações da prática (qualidade dos resultados, comparações e conclusões do grupo).

---

## 🏋️ Exercícios da Aula 04

Os quatro exercícios praticam context engineering no chatbot do grupo — posição das instruções críticas, compactação com XML tagging, teste A/B de aderência e meta prompting, com medição de tokens via tiktoken. Complete os andaimes, rode cada célula no Colab e registre o veredito como parte da entrega do CKP01.


### Exercício 1 — Context rot no código: regra no meio vs na borda · ★★☆ · 10 min

*Individual · Colab*

O paper "Lost in the Middle" mostra que o modelo concentra a atenção nas bordas do contexto. Prove com o mesmo teste rodado duas vezes.

1. Complete a regra crítica de redirecionamento do domínio (a MESMA string nas duas versões).
2. Preencha a tag `<critico>` na versão XML — a regra crítica fica na última borda do contexto.
3. Rode a célula e compare as duas respostas para a pergunta mista: qual versão redirecionou o tema de fora?
4. Em comentário: em que posição do contexto você colocaria as regras críticas no chatbot do CKP01?

> **💡 Dica:** na versão misturada a regra fica no meio de 8 instruções — a posição que o modelo tende a "esquecer".


In [ ]:
# 👉 LACUNA: a mesma regra crítica em duas posições — qual adere melhor?
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 👉 LACUNA 1: regra crítica de redirecionamento do domínio
# (ex.: "Se a pergunta sair do domínio, redirecione na primeira frase.")
REGRACRITICA = "___"

# Versão MISTURADA: a regra crítica no MEIO de 8 instruções
PROMPT_MISTURADO = f"""Você é um assistente de culinária. Responda sempre em português.
{REGRACRITICA}
Seja amigável e use emojis. Mencione ingredientes locais. Limite respostas a
3 parágrafos. Não use markdown. Cite fontes quando possível."""

# Versão XML: a regra crítica na ÚLTIMA borda, dentro de uma tag
# 👉 LACUNA 2: a MESMA regra crítica aqui dentro
PROMPT_XML = """<persona>Chef de culinária brasileira.</persona>
<formato>Sem markdown. Máximo 3 parágrafos.</formato>
<critico>___</critico>"""

llm = ChatOllama(model="gpt-oss:120b")
pergunta_teste = "Me explique o que é Docker. E aproveita: qual é a capital do Brasil?"

for nome, system in [("MISTURADA (regra no meio)", PROMPT_MISTURADO),
                     ("XML (regra na borda)", PROMPT_XML)]:
    chain = ChatPromptTemplate.from_messages([
        ("system", system),
        ("human",  "{pergunta}"),
    ]) | llm | StrOutputParser()
    print(f"=== {nome} ===")
    print(chain.invoke({"pergunta": pergunta_teste}), "\n")
# 👉 LACUNA 3: em comentário — qual versão redirecionou o tema de fora?


### Exercício 2 — XML tagging + tiktoken: comprima sem perder instruções · ★★☆ · 10 min

*Individual · Colab*

Compacte o system prompt do grupo com XML tagging e meça o ganho em tokens.

1. Cole o `SYSTEM_ORIGINAL` do grupo (Aulas 01–03) na primeira lacuna.
2. Reescreva com XML tagging — no mínimo `<persona>`, `<restricoes>` e `<formato>`, na forma imperativa, sem redundâncias.
3. Rode a medição: a redução chegou aos 30%? Em comentário, qual instrução essencial foi cortada (ou mantida)?

> **💡 Dica:** forma imperativa comprime — "Seja sempre amigável com o usuário" → "Tom amigável"; "Não fale sobre assuntos não relacionados" → "Somente [domínio]".


In [ ]:
# 👉 LACUNA: compacte o system prompt do grupo com XML tagging e meça o ganho
import tiktoken

def contar_tokens(texto: str, modelo: str = "gpt-4") -> int:
    return len(tiktoken.encoding_for_model(modelo).encode(texto))

# 👉 LACUNA 1: cole o SYSTEM_ORIGINAL do grupo (Aulas 01–03)
SYSTEM_ORIGINAL = ___

# 👉 LACUNA 2: reescreva com XML tagging — 3 tags no mínimo, forma imperativa
# (<persona>, <restricoes>, <formato> — e uma 4ª tag se fizer sentido)
SYSTEM_OTIMIZADO = ___

tok_antes  = contar_tokens(SYSTEM_ORIGINAL)
tok_depois = contar_tokens(SYSTEM_OTIMIZADO)
print(f"Antes:  {tok_antes} tokens")
print(f"Depois: {tok_depois} tokens")
print(f"Redução: {(1 - tok_depois / tok_antes) * 100:.1f}%")   # meta: ≥ 30%
# 👉 LACUNA 3: em comentário — qual instrução essencial você cortou (ou manteve)?


### Exercício 3 — Teste A/B: aderência antes e depois do contexto · ★★☆ · 10 min

*Individual · Colab*

Duas chains, mesmo modelo, mesma pergunta — a única variável é o system prompt.

1. Complete o mesmo placeholder `{pergunta}` nas duas mensagens human.
2. Escolha UMA pergunta mista (pedido do domínio + tema de fora) e invoque as duas chains com ela.
3. Em comentário: qual versão redirecionou a pergunta fora do domínio e respeitou o formato pedido?

> **💡 Dica:** uma única variável `pergunta_teste` usada nas duas chamadas garante que a comparação isole a diferença do prompt.


In [ ]:
# 👉 LACUNA: A/B da mesma pergunta antes vs depois do context engineering
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chain_antes = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_ORIGINAL),
    ("human",  "___"),            # 👉 LACUNA: mesmo placeholder nas duas chains
]) | llm | StrOutputParser()

chain_depois = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_OTIMIZADO),
    ("human",  "___"),
]) | llm | StrOutputParser()

# 👉 LACUNA: 1 pergunta do domínio + 1 de fora, na MESMA string
pergunta_teste = ___

print("=== ANTES ===\n",  chain_antes.invoke({"pergunta": ___}))
print("=== DEPOIS ===\n", chain_depois.invoke({"pergunta": ___}))
# 👉 LACUNA: em comentário — qual versão redirecionou a pergunta fora do domínio?


### Exercício 4 — Meta prompting: o modelo otimiza o próprio prompt · ★★☆ · 10 min

*Individual · Colab*

O modelo reescreve o system prompt do grupo aplicando context engineering.

1. Complete a chain otimizadora — template, modelo e parser na ordem certa.
2. Complete o `.invoke()` com a variável correta e o `SYSTEM_ORIGINAL` do grupo.
3. Extraia o texto entre `<prompt_otimizado>...</prompt_otimizado>` e meça os tokens com `contar_tokens()` — houve redução? Alguma instrução essencial se perdeu?

> **💡 Dica:** defina o `<formato_saida>` com precisão — peça o prompt entre tags e a explicação depois delas, para extrair os dois com `split` sem ambiguidade.


In [ ]:
# 👉 LACUNA: meta prompting — o modelo reescreve o prompt do grupo
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

PROMPT_OTIMIZADOR = """
<tarefa>
Você é um especialista em context engineering. Reescreva o system prompt
aplicando: XML tagging, instruções críticas no início e no final e remoção
de redundâncias.
</tarefa>

<prompt_original>
{prompt_original}
</prompt_original>

<formato_saida>
Retorne o prompt otimizado entre as tags <prompt_otimizado> e </prompt_otimizado>.
Depois liste em 3 bullet points as principais mudanças.
</formato_saida>
"""

chain_otimizadora = ___ | ___ | StrOutputParser()   # 👉 LACUNA

resultado = chain_otimizadora.invoke({"___": ___})  # 👉 LACUNA
print(resultado)

# 👉 LACUNA: extraia o texto entre <prompt_otimizado>...</prompt_otimizado>
# (use resultado.split(...) ou re.search) e meça os tokens com contar_tokens()


## 📚 Referências da aula

- Blog Anthropic Engineering — "Effective Context Engineering for AI Agents" (setembro, 2025). A fonte primária do termo e das técnicas desta aula. anthropic.com/engineering/building-effective-agents
- Paper Liu, N. et al. — "Lost in the Middle: How Language Models Use Long Contexts." EMNLP, 2023. Base empírica do context rot. arxiv.org/abs/2307.03172
- Docs Anthropic — Prompt Library e guia de XML tagging. docs.anthropic.com/pt/docs/build-with-claude/prompt-engineering/use-xml-tags
- Docs tiktoken — Biblioteca de tokenização da OpenAI. Funciona como aproximação para qualquer modelo baseado em BPE. github.com/openai/tiktoken
- Livro Russell, S.; Norvig, P. — Inteligência Artificial. 3ª ed. Pearson, 2016. Cap. 22 — A importância do contexto na inferência linguística: a base teórica de por que o contexto é informação.

---

**→ Próxima Aula — Aula 05 · 31/08** — Embeddings e busca semântica com ChromaDB
  
Transformar texto em vetores e buscar por similaridade. A fundação do RAG começa aqui.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*